In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
gold_loans = pd.read_csv('/data/gold_loans.csv')
gold_rates= pd.read_csv('/data/gold_rates.csv')
customers= pd.read_csv('/data/customers.csv')
repayments= pd.read_csv('/data/repayments.csv')
branches= pd.read_csv('/data/branches.csv')


In [ ]:
#How many rows are in gold_loans exactly as received (before any cleaning)?

gold_loans.shape


In [ ]:
#How many rows are in repayments?

repayments.shape

In [ ]:
#How many branches does Suvarna Finance have?

branches['branch_id'].nunique()

In [ ]:
#In how many different states does the company have branches?

branches['state'].nunique()

In [ ]:
#Return one row with the earliest and latest disbursal dates as first_disbursal and last_disbursal

dates=pd.to_datetime(gold_loans['disbursal_date'])
answer=pd.DataFrame({'first_disbursal':[dates.min()],'last_disbursal':[dates.max()]})
answer


In [ ]:
#How many distinct loan_id values are there?

gold_loans['loan_id'].nunique()

In [ ]:
#How many exact duplicate rows must be removed from gold_loans?

gold_loans.duplicated().sum()

In [ ]:
#In how many different ways is status written in the raw data (count distinct raw values, letter case included)?

gold_loans['status'].nunique()

In [ ]:
#After removing duplicates and converting status to lower case, return status and loans (the number of loans) for each status.

clean_data=gold_loans.drop_duplicates().copy()
clean_data['status']=clean_data['status'].str.strip().str.lower()
clean_data['principal']=clean_data['principal'].astype(float)
clean_data['gold_weight_gm']=pd.to_numeric(clean_data['gold_weight_gm'])
answer=clean_data.groupby('status')['loan_id'].count().reset_index(name='loans')
answer

In [ ]:
#How many different raw values does the purity column contain?

clean_data['purity'].nunique()

In [ ]:
#After removing duplicates and converting purity to a karat number (916 = 22, 833 = 20, 750 = 18, 999 = 24), return karat and loans for each karat.

clean_data=clean_data.drop_duplicates()
clean_data['karat']=(clean_data['purity'].str.replace('KT','')
.str.replace('K','')
.str.strip().replace({'916':22,'833':20,'750':18,'999':24}).astype(int))
answer=clean_data.groupby('karat')['loan_id'].count().reset_index(name='loans')
answer



In [ ]:
#After removing duplicates, how many loans have no gold weight recorded?

clean_data=clean_data.drop_duplicates()
clean_data['gold_weight_gm'].isnull().sum()

In [ ]:
#After removing duplicates, how many loans have a principal of 0 or less?

clean_data=clean_data.drop_duplicates()
answer=len(clean_data[clean_data['principal']<=0])
answer


In [ ]:
#How many active loans are there after removing duplicates (status 'active' in any letter case)?

answer=len(clean_data[clean_data['status']=='active'])
answer

In [ ]:
#How many active loans cannot be assessed because the gold weight is missing or the principal is 0 or less?

answer=((clean_data['status']=='active') & ((clean_data['principal']<=0) | (clean_data['gold_weight_gm'].isnull()))).sum()
answer

In [ ]:
#How many daily rates are in gold_rates?

gold_rates['rate_date']=pd.to_datetime(gold_rates['rate_date']).drop_duplicates()
answer=len(gold_rates[gold_rates['rate_date']!='sunday'])

answer

In [ ]:
#What was the highest 24K gold rate (₹ per gram) during 2026?

peak_rate=gold_rates['rate_24k_per_gram'].max()
peak_rate

In [ ]:
#Return one row with peak_date: the date of the highest 24K rate in 2026.

peak_rate=gold_rates['rate_24k_per_gram'].max()
answer=gold_rates.loc[gold_rates['rate_24k_per_gram']==peak_rate,['rate_date']]
answer=answer.rename(columns={'rate_date':'peak_date'})
answer

In [ ]:
#Which 24K rate (₹ per gram) applies on 15 October 2026?

gold_rates['rate_date']=pd.to_datetime(gold_rates['rate_date'])
gold_rates['rate_date'].dtype

assessment_date_rate=gold_rates.loc[gold_rates['rate_date']==pd.to_datetime('2026-10-15'),['rate_24k_per_gram']]
assessment_date_rate


In [ ]:
#By what percentage has the 24K rate fallen from its 2026 peak to the assessment date? Round to 2 decimals.

fall=round((peak_rate-assessment_date_rate)/peak_rate*100,2)
fall

In [ ]:
#How many distinct city names appear in branches? (Old names such as Bangalore count separately.)

branches['city'].nunique()

In [ ]:
#After removing duplicates, return scheme and loans for each scheme.

clean_data = clean_data.drop_duplicates()
answer=clean_data.groupby('scheme')['loan_id'].count().reset_index(name='loans')
answer




In [ ]:
#After removing duplicates, what is the total principal disbursed across all loans with a principal above 0 (₹)?

total_principal=clean_data['principal'].sum()
total_principal

In [ ]:
#What is the average principal (₹, rounded to 0 decimals) of assessable active loans?

assessible_loans=clean_data[(clean_data['principal']>0) & (clean_data['status']=='active') & (clean_data['gold_weight_gm'].notnull())].copy()
round(assessible_loans['principal'].mean(),0)

In [ ]:
#How many customers are in customers?

customers['customer_id'].nunique()

In [ ]:
#What is the average customer age? Round to 1 decimal.

round(customers['age'].mean(),1)


In [ ]:
#After removing duplicates, how many loans were disbursed in 2026?

clean_data['disbursal_date']=pd.to_datetime(clean_data['disbursal_date'])
loans_2026=clean_data[clean_data['disbursal_date'].dt.year==2026]
loans_2026['loan_id'].nunique()


In [ ]:
#After removing duplicates, return scheme and avg_rate (average interest_rate_pct, rounded to 2 decimals).

answer=round(clean_data.groupby('scheme')['interest_rate_pct'].mean(),2).reset_index(name='avg_rate')
answer

In [ ]:
#How many repayments have payment_type = 'principal'?

repayment_type_principal=repayments[repayments['payment_type']=='principal']
repayment_type_principal['payment_id'].nunique()

In [ ]:
#What is the total amount received as interest (₹)?

repayment_type_interest=repayments[repayments['payment_type']=='interest']
repayment_type_interest['amount'].sum()

In [ ]:
#What is the total outstanding principal (₹) of all assessable active loans? Outstanding = principal − principal repayments.
 
assessible_loans = clean_data[
    (clean_data['principal'] > 0) &
    (clean_data['status'] == 'active') &
    (clean_data['gold_weight_gm'].notnull())]

x = assessible_loans['principal'].sum()

principal_repayments = repayments[
    (repayments['payment_type'] == 'principal') & (repayments['loan_id'].isin(assessible_loans['loan_id']))]

y= principal_repayments['amount'].sum()

outstanding = x - y
outstanding

In [ ]:
#What is the average current LTV (%) of assessable active loans? Round to 2 decimals

# 1. Remove duplicate loans
assessible_loans = clean_data.drop_duplicates().copy()

assessible_loans = assessible_loans[
    (assessible_loans['principal'] > 0) &
    (assessible_loans['status'] == 'active') &
    (assessible_loans['gold_weight_gm'].notna())
].copy()

# 2. Convert rate_date into datetime
gold_rates['rate_date'] = pd.to_datetime(gold_rates['rate_date'])

# 3. Set reference date
R = pd.to_datetime('2026-10-15')

# 4. Find latest available gold rate before R
latest_rate = gold_rates.loc[
    gold_rates['rate_date'] <= R,
    ['rate_date', 'rate_24k_per_gram']
].sort_values('rate_date').iloc[-1]

# 5. Calculate total principal repayment per loan
repaid = repayments[
    repayments['payment_type'] == 'principal'
].groupby('loan_id')['amount'].sum()


# 6. Calculate outstanding principal
assessible_loans['outstanding'] = (
    assessible_loans['principal'] -
    assessible_loans['loan_id'].map(repaid).fillna(0)
)

# 7. Calculate current gold value
assessible_loans['current_gold_rate'] = (
    assessible_loans['gold_weight_gm'] *
    (assessible_loans['karat'] / 24) *
    latest_rate['rate_24k_per_gram']
)

# 8. Calculate current LTV (%)
assessible_loans['current_ltv_%'] = (
    assessible_loans['outstanding'] /
    assessible_loans['current_gold_rate']
) * 100

# 9. cal avg current ltv(%)

round(assessible_loans['current_ltv_%'].mean(),2)


In [ ]:
#How many assessable active loans are above the 75% LTV cap?

breach_loans=assessible_loans[ assessible_loans['current_ltv_%']>=75]
breach_loans.shape

In [ ]:
#What is the outstanding principal (₹) of loans above 75% LTV?

breach_loans['principal'].sum()


In [ ]:
#What percentage of assessable active loans are in breach? Round to 2 decimals.

percentage=round(len(breach_loans)/len(assessible_loans)*100,2)
percentage

In [ ]:
#Return scheme, breach_loans and active_loans (assessable active loans) for each scheme.

answer = assessible_loans.groupby('scheme').agg(
    breach_loans=('current_ltv_%', lambda x: (x > 75).sum()),
    active_loans=('loan_id', 'count')
).reset_index()
assessible_loans.groupby('scheme')['principal'].agg(['min','max','sum'])
answer

import numpy as np
import matplotlib.pyplot as plt

plt.figure(figsize=(9,5))

x = np.arange(len(answer))
width = 0.35

b1 = plt.bar(x-width/2, answer['active_loans'], width, label='Active Loans')
b2 = plt.bar(x+width/2, answer['breach_loans'], width, label='Breached Loans')

for i, j in zip(b1, answer['active_loans']):
    plt.text(i.get_x()+i.get_width()/2, i.get_height()+100, int(j), ha='center')

for i, j in zip(b2, answer['breach_loans']):
    plt.text(i.get_x()+i.get_width()/2, i.get_height()+100, int(j), ha='center')

plt.xticks(x, answer['scheme'])
plt.title('Scheme-wise LTV Breach Analysis')
plt.xlabel('Scheme')
plt.ylabel('Number of Loans')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#Return state and breach_outstanding (₹, rounded to 0) for every state with breaches, highest first.

answer=branches.merge(breach_loans, on='branch_id').groupby('state')['outstanding'].sum().reset_index(name='breach_outstanding').sort_values(by='breach_outstanding',ascending=False)
answer


import matplotlib.pyplot as plt

# Sort ascending so the highest value appears at the top of a horizontal chart
plot_data = answer.sort_values(by='breach_outstanding', ascending=True)
values_cr = plot_data['breach_outstanding'] / 1e7

plt.figure(figsize=(8, 5))
bars = plt.barh(plot_data['state'], values_cr, color='royalblue', height=0.6)

plt.title('State-Wise Breached Outstanding (in ₹ Crores)', fontsize=12, fontweight='bold')
plt.xlabel('Breach Outstanding (₹ Cr)', fontsize=10, fontweight='bold')
plt.ylabel('State', fontsize=10, fontweight='bold')

# Add extra space on the right so data labels don't touch the border
plt.xlim(0, values_cr.max() * 1.18)

# Add clean labels at the end of each bar
for bar in bars:
    width = bar.get_width()
    plt.text(width + (values_cr.max() * 0.02), bar.get_y() + bar.get_height()/2,
             f'₹{width:.2f} Cr', va='center', ha='left', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
#For breaching loans, return month (text 'YYYY-MM' of the disbursal date) and breach_loans, in month order.

answer=breach_loans.groupby(breach_loans['disbursal_date'].dt.strftime  ('%Y-%m'))['loan_id'].count().reset_index(name='breach_loans').sort_values(by='disbursal_date')
answer=answer.rename(columns={'disbursal_date':'month'})
answer

plt.figure(figsize=(5,5))
plt.plot(answer['month'],answer['breach_loans'],color='green',linestyle='--',marker='o',markersize=14,mfc='yellow',mec='red')
plt.title('Monthly Breached Loans',weight='bold')
plt.xlabel('Month',weight='bold')
plt.ylabel('Breach_loans',weight='bold')
plt.xticks(rotation=45,weight='bold')
plt.yticks(weight='bold')
for i,j in zip (answer['month'],answer['breach_loans']):
    plt.text(i,j+0.2,str(j),ha='center',va='bottom',fontsize=10)
plt.show()



In [ ]:
#Return the top 10 branches by breached outstanding: branch_id, branch_name, breach_outstanding (₹, rounded to 0), highest first.


answer=breach_loans[['branch_id','outstanding']].merge(branches[['branch_id','branch_name']], left_on='branch_id',right_on='branch_id', how='inner')
answer=answer.groupby(['branch_id','branch_name'])['outstanding'].sum().reset_index(name='breach_outstanding').sort_values(by='breach_outstanding',ascending=False).head(10)

answer



plt.figure(figsize=(8, 4))

labels = answer['branch_id'] + ' - ' + answer['branch_name']

plt.barh(labels, answer['breach_outstanding'], color='teal', height=0.7)
plt.gca().invert_yaxis()

plt.title('Branch Wise Breached Outstanding', weight='bold')
plt.ylabel('Branch', fontsize=10, weight='bold')
plt.xlabel('Breach Outstanding', fontsize=10, weight='bold')

for i, j in zip(labels, answer['breach_outstanding']):
    plt.text(j / 2, i, f'{j:,.0f}', ha='center', va='center', color='white', fontweight='bold', fontsize=9)


In [ ]:
#What percentage of assessable active Express loans are in breach? Round to 1 decimal.

T1=assessible_loans[assessible_loans['scheme']=='Express']
T1=T1['loan_id'].count()
T1

T2=breach_loans[breach_loans['scheme']=='Express']
T2=T2['loan_id'].count()
T2

answer=round((T2/T1)*100,1)
answer

In [ ]:
#Return region, active_loans and outstanding (₹, rounded to 0) for assessable active loans in each region.


answer=branches[['region','branch_id']].merge(assessible_loans[['loan_id','outstanding','branch_id']],left_on='branch_id',right_on='branch_id',how='inner')
answer=answer.groupby('region').agg(active_loans=('loan_id','count'),outstanding=('outstanding','sum')
).reset_index()
answer['outstanding']=answer['outstanding'].round(0)
answer


In [ ]:
#For loans disbursed from 1 July to 30 September 2026 (after removing duplicates), return scheme and avg_ltv_at_disbursal (rounded to 2).

date_data=clean_data[(clean_data['disbursal_date']>='2026-07-01') & (clean_data['disbursal_date']<='2026-09-30')]
answer=date_data.groupby('scheme')['ltv_at_disbursal_pct'].mean().round(2).reset_index(name='avg_ltv_at_disbursal')
answer


In [ ]:
#How many active loans (after removing duplicates) are past their maturity date (disbursal date + tenure months) on 15 October 2026?

assessible_loans=assessible_loans.drop_duplicates()
assessible_loans['maturity_date']=assessible_loans.apply(lambda x:x['disbursal_date']+pd.DateOffset(months=int(x['tenure_months'])),axis=1)
x=assessible_loans[assessible_loans['maturity_date']<'2026-10-15']
x['loan_id'].count()


In [ ]:
#How many distinct customers have at least one loan in breach?

breach_loans['customer_id'].nunique()

In [ ]:
#Return occupation and breach_loans, highest first.

answer = customers.merge(breach_loans, on='customer_id')
answer=answer.groupby('occupation')['loan_id'].count().reset_index(name='breach_loans').sort_values(by='breach_loans', ascending=False)

answer

# Use a separate variable so 'answer' stays sorted highest-first for grading
plot_df = answer.sort_values(by='breach_loans', ascending=True)

plt.figure(figsize=(8, 4.5))
plt.barh(plot_df['occupation'], plot_df['breach_loans'], color='pink', height=0.6)

plt.title('Occupation-Wise Breached Loans', fontsize=12, fontweight='bold')
plt.xlabel('Breach Loans', fontsize=10, fontweight='bold')
plt.ylabel('Occupation', fontsize=10, fontweight='bold')
plt.xticks(weight='bold')
plt.yticks(weight='bold')

# Add space on the right so the top label (673) fits inside the frame
plt.xlim(0, plot_df['breach_loans'].max() * 1.15)

# Place exact counts neatly to the right of each bar
for idx, val in enumerate(plot_df['breach_loans']):
    plt.text(val + 10, idx, str(val), va='center', ha='left', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
#If the gold rate falls a further 5% from the assessment-date rate, how many assessable active loans would be above 75% LTV?

assessible_loans['new_gold_rate_5%deduction']=assessible_loans['current_gold_rate']*0.95
assessible_loans['new_ltv_%']=(assessible_loans['outstanding']/assessible_loans['new_gold_rate_5%deduction'])*100
result=assessible_loans['new_ltv_%']>75
result.sum()

In [ ]:
#How much principal (₹, rounded to 0) must be repaid in total to bring every breaching loan back to exactly 75% LTV?

breach_loans['outstanding_at75%_ltv']=breach_loans['current_gold_rate']*0.75
breach_loans['repay']=round(breach_loans['outstanding']-breach_loans['outstanding_at75%_ltv'],0)
breach_loans['repay'].sum()



In [ ]:
#How many assessable active loans are above 80% LTV (critical)?

(assessible_loans['current_ltv_%']>80).sum()

In [ ]:
#For each region, return the branch with the highest breach rate (% of its assessable active loans in breach): region, branch_name, breach_rate_pct (rounded to 2).

assessible_loans['breach_new']=assessible_loans['current_ltv_%']>75
breaching=assessible_loans.loc[assessible_loans['breach_new']]
(breaching['outstanding']-75 * breaching['current_gold_rate']).sum()
branch_breach=assessible_loans.merge(branches,on='branch_id').groupby(['region','branch_name']).agg(breaching=('breach_new','sum'),clean_data=('loan_id','size')).reset_index()
branch_breach['breach_rate_pct']=round(branch_breach['breaching']/branch_breach['clean_data']*100,2)
answer=branch_breach[['region','branch_name','breach_rate_pct']]
answer=answer.sort_values(['region','breach_rate_pct','branch_name'],ascending=[True,False,True]).groupby('region').head(1)



answer

In [ ]:
#For breaching loans disbursed in 2026, return month ('YYYY-MM'), breach_outstanding (rounded to 0) and cumulative_outstanding (running total, rounded to 0), in month order.

cumulative_outstanding=breaching.loc[breaching['disbursal_date'].dt.year==2026].assign(month=lambda x:x['disbursal_date'].dt.strftime('%Y-%m')).groupby('month',as_index=False)['outstanding'].sum().sort_values('month')

cumulative_outstanding['breach_outstanding']=cumulative_outstanding['outstanding'].round(0)
cumulative_outstanding['cumulative_outstanding']=cumulative_outstanding['breach_outstanding'].cumsum()

answer=cumulative_outstanding[['month','breach_outstanding','cumulative_outstanding']]
answer

In [ ]:


labels = ['Below 75%', '75%-80%', 'Above 80%']
values = [16273, 1938, 564]

plt.figure(figsize=(7,7))

plt.pie(values, labels=labels, autopct='%1.1f%%',
        startangle=90, wedgeprops={'width':0.4})

plt.title('LTV Breach Distribution', weight='bold')
plt.show()